In [15]:
import torch
import numpy as np
import random
import os
import json
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.base import clone

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    balanced_accuracy_score
)
from torchmetrics.classification import MulticlassCalibrationError

# --- Configurazione ---
CURRENT_SEED = 17 # 11, 17, 29
COMPONENTS   = [32, 16, 8, 4]
N_CLASSES    = 4
# ----------------------

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CURRENT_SEED)
os.makedirs('../artifacts', exist_ok=True)

print(f'Seed: {CURRENT_SEED}')
print(f'Regimi latenti: {COMPONENTS}')

Seed: 17
Regimi latenti: [32, 16, 8, 4]


In [3]:
def load_pca_split(d, seed):

    path = f'../compressedFeatures/pca_d{d}_seed_{seed}.pt'
    ckpt = torch.load(path, weights_only=False)

    X_train = ckpt['train'].numpy()
    X_val   = ckpt['val'].numpy()
    X_test  = ckpt['test'].numpy()

    # squeeze per sklearn per problemi tra medmnist (N, 1) e sklearn (N, )
    y_train = ckpt['y_train'].numpy().squeeze()
    y_val   = ckpt['y_val'].numpy().squeeze()
    y_test  = ckpt['y_test'].numpy().squeeze()

    print(f'd={d:2d} | train={X_train.shape}, val={X_val.shape}, test={X_test.shape}')
    return X_train, X_val, X_test, y_train, y_val, y_test

for d in COMPONENTS:
    load_pca_split(d, CURRENT_SEED)

d=32 | train=(4000, 32), val=(1000, 32), test=(1000, 32)
d=16 | train=(4000, 16), val=(1000, 16), test=(1000, 16)
d= 8 | train=(4000, 8), val=(1000, 8), test=(1000, 8)
d= 4 | train=(4000, 4), val=(1000, 4), test=(1000, 4)


- **Macro-AUROC**: calcolata in multiclass one-vs-rest sulle probabilità di classe
- **Macro-F1**: calcolata sulle label ottenute via argmax
- **Balanced Accuracy**: calcolate sulle label ottenute via argmax
- **ECE**:  top-label multiclass calibration error con 15 bin uniformi e norma L1

In [4]:
def compute_metrics(y_true, y_pred, probs):

    macro_auroc = roc_auc_score(
        y_true, probs,
        multi_class='ovr',
        average='macro'
    )

    macro_f1 = f1_score(
        y_true, y_pred,
        average='macro',
        zero_division=0
    )

    bal_acc = balanced_accuracy_score(y_true, y_pred)

    ece_metric = MulticlassCalibrationError(
        num_classes=N_CLASSES,
        n_bins=15,
        norm='l1'
    )
    ece = ece_metric(
        torch.tensor(probs, dtype=torch.float32),
        torch.tensor(y_true, dtype=torch.long)
    ).item()

    return {
        'macro_auroc': round(macro_auroc, 4),
        'macro_f1':    round(macro_f1,    4),
        'bal_acc':     round(bal_acc,     4),
        'ece':         round(ece,         4)
    }

In [5]:
def run_logi_regr(X_train, y_train, X_val, y_val, X_test, y_test, d):

    clf = LogisticRegression(
        solver='lbfgs',
        max_iter=1000,
        random_state=CURRENT_SEED
    )
    clf.fit(X_train, y_train)

    # Validation
    val_probs  = clf.predict_proba(X_val)
    val_pred   = val_probs.argmax(axis=1)
    val_metrics = compute_metrics(y_val, val_pred, val_probs)

    # Test
    test_probs  = clf.predict_proba(X_test)
    test_pred   = test_probs.argmax(axis=1)
    test_metrics = compute_metrics(y_test, test_pred, test_probs)

    return clf, val_metrics, test_metrics, val_pred, test_pred, val_probs, test_probs

all_results = {}

for d in COMPONENTS:
    X_train, X_val, X_test, y_train, y_val, y_test = load_pca_split(d, CURRENT_SEED)

    clf, val_m, test_m, val_pred, test_pred, val_probs, test_probs = run_logi_regr(X_train, y_train, X_val, y_val, X_test, y_test, d)

    all_results[d] = {
        'val':  val_m,
        'test': test_m
    }

    print(f'd={d:2d}')
    print(f'  VAL  → AUROC={val_m["macro_auroc"]:.4f} | F1={val_m["macro_f1"]:.4f} | BalAcc={val_m["bal_acc"]:.4f} | ECE={val_m["ece"]:.4f}')
    print(f'  TEST → AUROC={test_m["macro_auroc"]:.4f} | F1={test_m["macro_f1"]:.4f} | BalAcc={test_m["bal_acc"]:.4f} | ECE={test_m["ece"]:.4f}\n')

d=32 | train=(4000, 32), val=(1000, 32), test=(1000, 32)
d=32
  VAL  → AUROC=0.9387 | F1=0.7935 | BalAcc=0.7930 | ECE=0.0362
  TEST → AUROC=0.9469 | F1=0.8154 | BalAcc=0.8150 | ECE=0.0872

d=16 | train=(4000, 16), val=(1000, 16), test=(1000, 16)
d=16
  VAL  → AUROC=0.9109 | F1=0.7181 | BalAcc=0.7180 | ECE=0.0314
  TEST → AUROC=0.8652 | F1=0.6454 | BalAcc=0.6560 | ECE=0.0469

d= 8 | train=(4000, 8), val=(1000, 8), test=(1000, 8)
d= 8
  VAL  → AUROC=0.8657 | F1=0.6295 | BalAcc=0.6300 | ECE=0.0324
  TEST → AUROC=0.8513 | F1=0.5524 | BalAcc=0.5730 | ECE=0.0560

d= 4 | train=(4000, 4), val=(1000, 4), test=(1000, 4)
d= 4
  VAL  → AUROC=0.8388 | F1=0.6081 | BalAcc=0.6090 | ECE=0.0347
  TEST → AUROC=0.8507 | F1=0.5990 | BalAcc=0.6050 | ECE=0.0577



In [19]:
for d in COMPONENTS:
    X_train, X_val, X_test, y_train, y_val, y_test = load_pca_split(d, CURRENT_SEED)

    clf, val_m, test_m, val_pred, test_pred, val_probs, test_probs = \
        run_logi_regr(X_train, y_train, X_val, y_val, X_test, y_test, d)

    artifact = {
        # identificazione run
        'd':           d,
        'seed':        CURRENT_SEED,
        'model':       'logistic_regression',
        'compression': 'pca_baseline',
        'type':        'no_quantum_ablation',

        # predizioni
        'val_y_true':  y_val.tolist(),
        'val_y_pred':  val_pred.tolist(),
        'val_probs':   val_probs.tolist(),

        'test_y_true': y_test.tolist(),
        'test_y_pred': test_pred.tolist(),
        'test_probs':  test_probs.tolist(),

        # metriche
        'val_metrics':  val_m,
        'test_metrics': test_m
    }

    artifact = {
        'd': d, 'seed': CURRENT_SEED,
        'model': 'logistic_regression',
        'compression': 'pca_baseline',
        'type': 'no_quantum_ablation',
        'gridsearch': False,
        'val_metrics': val_m,
        'test_metrics': test_m
    }
    with open(f'../artifacts/no_quantum_ablation_d{d}_seed_{CURRENT_SEED}.json', 'w') as f:
        json.dump(artifact, f, indent=2)
    np.savez(
        f'../artifacts/no_quantum_ablation_d{d}_seed_{CURRENT_SEED}_preds.npz',
        val_y_true=y_val,   val_y_pred=val_pred,   val_probs=val_probs,
        test_y_true=y_test, test_y_pred=test_pred, test_probs=test_probs
    )


d=32 | train=(4000, 32), val=(1000, 32), test=(1000, 32)
d=16 | train=(4000, 16), val=(1000, 16), test=(1000, 16)
d= 8 | train=(4000, 8), val=(1000, 8), test=(1000, 8)
d= 4 | train=(4000, 4), val=(1000, 4), test=(1000, 4)


# MLP a un hidden layer

In [7]:
def run_mlp(X_train, y_train, X_val, y_val, X_test, y_test):
    clf = MLPClassifier(
        hidden_layer_sizes=(64,),
        activation='relu',
        solver='adam',
        max_iter=200,
        random_state=CURRENT_SEED,
        early_stopping=True,    
        validation_fraction=0.1
    )
    clf.fit(X_train, y_train)

    val_probs   = clf.predict_proba(X_val)
    val_pred    = val_probs.argmax(axis=1)
    val_metrics = compute_metrics(y_val, val_pred, val_probs)

    test_probs   = clf.predict_proba(X_test)
    test_pred    = test_probs.argmax(axis=1)
    test_metrics = compute_metrics(y_test, test_pred, test_probs)

    return clf, val_metrics, test_metrics, val_pred, test_pred, val_probs, test_probs

mlp_results = {}

for d in COMPONENTS:
    X_train, X_val, X_test, y_train, y_val, y_test = load_pca_split(d, CURRENT_SEED)

    clf, val_m, test_m, val_pred, test_pred, val_probs, test_probs = \
        run_mlp(X_train, y_train, X_val, y_val, X_test, y_test)

    mlp_results[d] = {'val': val_m, 'test': test_m}

    print(f'd={d:2d}')
    print(f'  VAL  → AUROC={val_m["macro_auroc"]:.4f} | F1={val_m["macro_f1"]:.4f} | BalAcc={val_m["bal_acc"]:.4f} | ECE={val_m["ece"]:.4f}')
    print(f'  TEST → AUROC={test_m["macro_auroc"]:.4f} | F1={test_m["macro_f1"]:.4f} | BalAcc={test_m["bal_acc"]:.4f} | ECE={test_m["ece"]:.4f}\n')

d=32 | train=(4000, 32), val=(1000, 32), test=(1000, 32)
d=32
  VAL  → AUROC=0.9429 | F1=0.7892 | BalAcc=0.7900 | ECE=0.0216
  TEST → AUROC=0.9427 | F1=0.8007 | BalAcc=0.8000 | ECE=0.0566

d=16 | train=(4000, 16), val=(1000, 16), test=(1000, 16)
d=16
  VAL  → AUROC=0.9150 | F1=0.7199 | BalAcc=0.7190 | ECE=0.0440
  TEST → AUROC=0.8923 | F1=0.6858 | BalAcc=0.6860 | ECE=0.0327

d= 8 | train=(4000, 8), val=(1000, 8), test=(1000, 8)
d= 8
  VAL  → AUROC=0.8759 | F1=0.6527 | BalAcc=0.6520 | ECE=0.0360
  TEST → AUROC=0.8433 | F1=0.5499 | BalAcc=0.5620 | ECE=0.0572

d= 4 | train=(4000, 4), val=(1000, 4), test=(1000, 4)
d= 4
  VAL  → AUROC=0.8413 | F1=0.6032 | BalAcc=0.6050 | ECE=0.0285
  TEST → AUROC=0.8722 | F1=0.6320 | BalAcc=0.6310 | ECE=0.0557



# SVM con RBF come kernel

In [8]:
def run_svm(X_train, y_train, X_val, y_val, X_test, y_test):
    clf = SVC(
        kernel='rbf',
        C=1.0,
        gamma='scale',
        probability=True,          # necessario per predict_proba e AUROC
        random_state=CURRENT_SEED,
        decision_function_shape='ovr'
    )
    clf.fit(X_train, y_train)

    val_probs    = clf.predict_proba(X_val)
    val_pred     = val_probs.argmax(axis=1)
    val_metrics  = compute_metrics(y_val, val_pred, val_probs)

    test_probs   = clf.predict_proba(X_test)
    test_pred    = test_probs.argmax(axis=1)
    test_metrics = compute_metrics(y_test, test_pred, test_probs)

    return clf, val_metrics, test_metrics, val_pred, test_pred, val_probs, test_probs

svm_results = {}

for d in COMPONENTS:
    X_train, X_val, X_test, y_train, y_val, y_test = load_pca_split(d, CURRENT_SEED)

    clf, val_m, test_m, val_pred, test_pred, val_probs, test_probs = \
        run_svm(X_train, y_train, X_val, y_val, X_test, y_test)

    svm_results[d] = {'val': val_m, 'test': test_m}

    print(f'd={d:2d}')
    print(f'  VAL  → AUROC={val_m["macro_auroc"]:.4f} | F1={val_m["macro_f1"]:.4f} | BalAcc={val_m["bal_acc"]:.4f} | ECE={val_m["ece"]:.4f}')
    print(f'  TEST → AUROC={test_m["macro_auroc"]:.4f} | F1={test_m["macro_f1"]:.4f} | BalAcc={test_m["bal_acc"]:.4f} | ECE={test_m["ece"]:.4f}\n')

d=32 | train=(4000, 32), val=(1000, 32), test=(1000, 32)
d=32
  VAL  → AUROC=0.9507 | F1=0.7996 | BalAcc=0.8000 | ECE=0.0280
  TEST → AUROC=0.9577 | F1=0.8225 | BalAcc=0.8230 | ECE=0.0505

d=16 | train=(4000, 16), val=(1000, 16), test=(1000, 16)
d=16
  VAL  → AUROC=0.9245 | F1=0.7392 | BalAcc=0.7400 | ECE=0.0309
  TEST → AUROC=0.9031 | F1=0.7089 | BalAcc=0.7090 | ECE=0.0427

d= 8 | train=(4000, 8), val=(1000, 8), test=(1000, 8)
d= 8
  VAL  → AUROC=0.8822 | F1=0.6584 | BalAcc=0.6590 | ECE=0.0385
  TEST → AUROC=0.8793 | F1=0.6189 | BalAcc=0.6220 | ECE=0.0573

d= 4 | train=(4000, 4), val=(1000, 4), test=(1000, 4)
d= 4
  VAL  → AUROC=0.8458 | F1=0.6220 | BalAcc=0.6240 | ECE=0.0252
  TEST → AUROC=0.8648 | F1=0.6345 | BalAcc=0.6390 | ECE=0.0523



# Grid Search

In [16]:
def run_gridsearch(estimator, param_grid, X_train, y_train, X_val, y_val):
    
    # combina train e val in un unico array
    X_all = np.concatenate([X_train, X_val], axis=0)
    y_all = np.concatenate([y_train, y_val], axis=0)

    # -1 = campione di train, 0 = campione di val
    split_index = np.concatenate([
        np.full(len(X_train), -1),
        np.zeros(len(X_val), dtype=int)
    ])
    ps = PredefinedSplit(split_index)

    gs = GridSearchCV(
        estimator,
        param_grid,
        cv=ps,
        scoring='f1_macro',
        refit=False,
        n_jobs=-1,     # usa tutti i core disponibili
        verbose=1
    )
    gs.fit(X_all, y_all)

    best_clf = clone(estimator).set_params(**gs.best_params_)
    best_clf.fit(X_train, y_train)

    return best_clf, gs.best_params_, gs.best_score_

In [17]:
# Griglie di parametri per ogni modello
lr_grid = {
    'C': [0.01, 0.1, 1.0, 10.0]
}
lr_estimator = LogisticRegression(
    solver='lbfgs',
    max_iter=1000,
    random_state=CURRENT_SEED
)

mlp_grid = {
    'hidden_layer_sizes': [(32,), (64,), (128,)],
    'alpha':              [0.0001, 0.001, 0.01]
}
mlp_estimator = MLPClassifier(
    activation='relu',
    solver='adam',
    max_iter=200,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=CURRENT_SEED
)

svm_grid = {
    'C':     [0.1, 1.0, 10.0],
    'gamma': ['scale', 'auto']
}
svm_estimator = SVC(
    kernel='rbf',
    probability=True,
    random_state=CURRENT_SEED
)

models_to_tune = [
    ('LogReg', lr_estimator, lr_grid),
    ('MLP',    mlp_estimator, mlp_grid),
    ('SVM',    svm_estimator, svm_grid)
]

In [21]:
gs_results = {}

for d in COMPONENTS:
    X_train, X_val, X_test, y_train, y_val, y_test = load_pca_split(d, CURRENT_SEED)
    gs_results[d] = {}

    print(f'\n{"="*55}\nd={d}')

    for name, estimator, grid in models_to_tune:
        print(f'\n--- {name} ---')

        best_clf, best_params, best_val_f1 = run_gridsearch(
            estimator, grid, X_train, y_train, X_val, y_val
        )
        print(f'Migliori parametri: {best_params}')
        print(f'Miglior F1 su val:  {best_val_f1:.4f}')

        # valuta sul test con il modello migliore
        val_probs    = best_clf.predict_proba(X_val)
        val_pred     = val_probs.argmax(axis=1)
        val_metrics  = compute_metrics(y_val, val_pred, val_probs)

        test_probs   = best_clf.predict_proba(X_test)
        test_pred    = test_probs.argmax(axis=1)
        test_metrics = compute_metrics(y_test, test_pred, test_probs)

        print(f'VAL  → AUROC={val_metrics["macro_auroc"]:.4f} | F1={val_metrics["macro_f1"]:.4f} | BalAcc={val_metrics["bal_acc"]:.4f} | ECE={val_metrics["ece"]:.4f}')
        print(f'TEST → AUROC={test_metrics["macro_auroc"]:.4f} | F1={test_metrics["macro_f1"]:.4f} | BalAcc={test_metrics["bal_acc"]:.4f} | ECE={test_metrics["ece"]:.4f}')

        gs_results[d][name] = {
            'best_params': best_params,
            'best_val_f1': best_val_f1,
            'val':  val_metrics,
            'test': test_metrics
        }

d=32 | train=(4000, 32), val=(1000, 32), test=(1000, 32)

d=32

--- LogReg ---
Fitting 1 folds for each of 4 candidates, totalling 4 fits
Migliori parametri: {'C': 1.0}
Miglior F1 su val:  0.7935
VAL  → AUROC=0.9387 | F1=0.7935 | BalAcc=0.7930 | ECE=0.0362
TEST → AUROC=0.9469 | F1=0.8154 | BalAcc=0.8150 | ECE=0.0872

--- MLP ---
Fitting 1 folds for each of 9 candidates, totalling 9 fits
Migliori parametri: {'alpha': 0.0001, 'hidden_layer_sizes': (128,)}
Miglior F1 su val:  0.7947
VAL  → AUROC=0.9470 | F1=0.7947 | BalAcc=0.7940 | ECE=0.0280
TEST → AUROC=0.9532 | F1=0.8038 | BalAcc=0.8030 | ECE=0.0505

--- SVM ---
Fitting 1 folds for each of 6 candidates, totalling 6 fits
Migliori parametri: {'C': 1.0, 'gamma': 'scale'}
Miglior F1 su val:  0.8037
VAL  → AUROC=0.9507 | F1=0.7996 | BalAcc=0.8000 | ECE=0.0280
TEST → AUROC=0.9577 | F1=0.8225 | BalAcc=0.8230 | ECE=0.0505
d=16 | train=(4000, 16), val=(1000, 16), test=(1000, 16)

d=16

--- LogReg ---
Fitting 1 folds for each of 4 candidates, to